# 🕵️ Dia 25 — Investigação em redes sociais, parte 1: reconhecimento por plataforma

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 25* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-25.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O `@serraazul.oficial` foi criado em março de 2025 como "Achadinhos BH", virou "Promoções Mineiras" e, em **4 de outubro**, ganhou o nome atual. Véspera do certificado do site falso? Rafa: *"A linha do tempo responde. Você lê. Não segue, não curte, não comenta, não manda 'oi' no WhatsApp."*

Neste notebook você vai:

1. **Montar consultas** do X e do Reddit e converter `created_utc`.
2. **Jogar "qual a precisão desta data?"**.
3. **Gerar o CSV do TimelineJS** e detectar a semana mais densa.
4. **Desenhar a linha do tempo em texto**, com a precisão visível.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. Consultas e carimbos de tempo

O notebook **monta** as consultas; você as roda à mão, logado, só lendo. Plataformas proíbem coleta automatizada nos termos de uso.

In [ ]:
#@title Consultas por plataforma { display-mode: "form" }
conta = "ExampleOrg"  #@param {type:"string"}
dominio = "example.org"  #@param {type:"string"}
inicio = "2025-01-01"  #@param {type:"date"}
fim = "2025-07-01"  #@param {type:"date"}

print("X (caixa de busca, logado; aba Mais recentes):")
for q in (f"from:{conta} since:{inicio} until:{fim}",        # until: é EXCLUSIVO
          f"from:{conta} filter:media",
          f"from:{conta} filter:links -filter:replies",
          f"to:{conta}",
          f"@{conta} -from:{conta}",
          f"url:{dominio}",
          f"from:{conta} min_retweets:100"):
    print("  ", q)
print("\nReddit (ordenar por Novos):")
for q in (f"site:{dominio}", f"url:{dominio}", f'"{conta}" subreddit:suacidade'):
    print("  ", q)

In [ ]:
from datetime import datetime, timezone, timedelta

def de_unix(carimbo):
    """created_utc do Reddit (segundos desde 1970, UTC) → texto legível em UTC e em Brasília."""
    utc = datetime.fromtimestamp(int(carimbo), timezone.utc)
    brt = utc.astimezone(timezone(timedelta(hours=-3)))          # BRT = UTC-3 (sem horário de verão)
    return utc.strftime("%Y-%m-%dT%H:%M:%SZ"), brt.strftime("%d/%m/%Y %H:%M BRT")

for c in (1735689600, 1731888000):                               # os dois exemplos do original
    print(c, "→", *de_unix(c))
print("\nMesmo que: date -u -d @1735689600  (Linux)  /  date -u -r 1735689600  (macOS)")

## 2. 🎮 Qual a precisão desta data?

1 = segundo · 2 = dia · 3 = mês · 4 = ano autodeclarado · 5 = visto pela primeira vez (não é data de criação)

In [ ]:
QUIZ = [
    ("'Joined September 2016' no perfil do X", "3", "O X só mostra mês e ano."),
    ("created_utc 1731888000 de um tópico do Reddit", "1", "Carimbo Unix: exato até o segundo."),
    ("'Fundada em 1998' na aba Sobre do LinkedIn", "4", "Informado pela própria organização: autodeclarado, uma fonte só."),
    ("Print do WhatsApp 'Serra Azul Atendimento' recebido pela ouvidoria em 5/out", "5",
     "O WhatsApp não mostra criação de conta. A data é do print: 'visto pela 1ª vez'."),
    ("Data de criação na Transparência da Página do Facebook", "2", "Dia exato, mas é afirmação da plataforma: uma fonte."),
]
MODO_JOGO = True
acertos = 0
nomes = ["segundo", "dia", "mês", "ano autodeclarado", "visto pela 1ª vez"]
for caso, certo, porque in QUIZ:
    print(f"\n• {caso}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3", "4", "5"):
            r = input("   Precisão (1-5): ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end="")
    print(f"   {nomes[int(certo) - 1]}: {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 3. CSV do TimelineJS e a semana mais densa

Eventos idênticos a [`caso-continuo/dados/dia-25/eventos-isca.csv`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-25).

In [ ]:
import csv
from collections import defaultdict
from datetime import date

# (data, precisão, título, texto com fonte, grupo)
EVENTOS = [
 [
  "1998",
  "ano (autodeclarado)",
  "Fundação da cooperativa",
  "LinkedIn 'Sobre', autodeclarado; registro #40",
  "Oficial"
 ],
 [
  "2012-05-14",
  "dia",
  "Página no Facebook criada",
  "Transparência da Página /serraazulcoop; registro #41",
  "Oficial"
 ],
 [
  "2015-03",
  "mês",
  "Instagram @serraazul criado",
  "Instagram 'Sobre esta conta' (mês e ano); registro #42",
  "Oficial"
 ],
 [
  "2016-08-02",
  "dia",
  "Canal @serraazulcoop no YouTube",
  "Data de entrada do canal; registro #43",
  "Oficial"
 ],
 [
  "2026-10-05",
  "dia",
  "Alerta oficial no Instagram @serraazul",
  "Post 'Não enviamos SMS com links'; registro #52",
  "Oficial"
 ],
 [
  "2025-03",
  "mês",
  "Conta 'Achadinhos BH' criada",
  "Instagram 'Sobre esta conta' do perfil hoje @serraazul.oficial; registro #44",
  "Instagram falso"
 ],
 [
  "2025-11-20",
  "dia",
  "Renomeada para 'Promoções Mineiras'",
  "Histórico de nomes de usuário; registro #44",
  "Instagram falso"
 ],
 [
  "2026-10-04",
  "dia",
  "Renomeada para @serraazul.oficial",
  "Histórico de nomes de usuário; registro #44",
  "Instagram falso"
 ],
 [
  "2026-10-04",
  "dia",
  "Anúncio impulsionado 'Desbloqueie seu Pix'",
  "Biblioteca de Anúncios (ativo 04 a 06/out), capturado 06/out; registro #45",
  "Instagram falso"
 ],
 [
  "2026-10-02",
  "dia",
  "serraazul-pix.example registrado",
  "RDAP (Dia 29 confirma); registro #20",
  "Domínio"
 ],
 [
  "2026-10-04",
  "dia",
  "Certificado TLS de serraazul-pix.example",
  "crt.sh; registro #21",
  "Domínio"
 ],
 [
  "2026-10-05T10:12:00Z",
  "segundo",
  "Disparo dos SMS (07h12 BRT)",
  "Logs da operadora entregues pela cooperativa; registro #3",
  "Domínio"
 ],
 [
  "2026-10-05",
  "visto pela 1ª vez",
  "WhatsApp 'Serra Azul Atendimento' visto",
  "Print enviado por cooperado à ouvidoria (a data é do print, não da criação); registro #47",
  "WhatsApp falso"
 ],
 [
  "2026-10-05",
  "dia",
  "37 reclamações na ouvidoria",
  "Relatório da ouvidoria; registro #4",
  "Reclamações"
 ],
 [
  "2026-10-06",
  "dia",
  "Primeira queixa pública no Reclame Aqui",
  "Data da publicação da queixa; registro #48",
  "Reclamações"
 ],
 [
  "2026-10-07T14:03:11Z",
  "segundo",
  "Tópico em fórum público sobre o SMS",
  "created_utc do tópico; registro #49",
  "Reclamações"
 ]
]

# Toda linha precisa citar o registro (Checkpoint do original)
sem_fonte = [e for e in EVENTOS if "registro #" not in e[3]]
assert not sem_fonte, f"Eventos sem linha do registro: {sem_fonte}"

COLS = ["Year", "Month", "Day", "Time", "End Year", "End Month", "End Day", "End Time",
        "Display Date", "Headline", "Text", "Media", "Media Credit", "Media Caption",
        "Media Thumbnail", "Type", "Group", "Background"]          # cabeçalho do modelo da Knight Lab
with open("timeline-isca.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLS)
    w.writeheader()
    for data, precisao, titulo, texto, grupo in EVENTOS:
        dia_parte = data[:10].split("-") + ["", ""]               # "2025-03" → ["2025","03","",""]
        hora = data[11:19] if "T" in data else ""
        w.writerow({"Year": dia_parte[0], "Month": dia_parte[1].lstrip("0"), "Day": dia_parte[2].lstrip("0"),
                    "Time": hora, "Headline": titulo, "Text": f"{texto} (precisão: {precisao})",
                    "Group": grupo})
print(f"💾 timeline-isca.csv: {len(EVENTOS)} eventos, {len({e[4] for e in EVENTOS})} grupos")

# Janela de 7 dias mais densa: a que tem mais GRUPOS diferentes ativos.
# Só entram datas com precisão de dia ou melhor (mês e ano não cabem numa semana).
from datetime import timedelta
datados = [(date.fromisoformat(d[:10]), g) for d, _p, _t, _x, g in EVENTOS if len(d) >= 10]
melhor = None
for inicio_j, _g in datados:                                     # cada evento pode abrir uma janela
    fim_j = inicio_j + timedelta(days=6)
    grupos = {g for d, g in datados if inicio_j <= d <= fim_j}
    if melhor is None or len(grupos) > len(melhor[2]):
        melhor = (inicio_j, fim_j, grupos)
ini, fim_, grupos = melhor
print(f"\nJanela de 7 dias mais densa: {ini:%d/%m/%Y} a {fim_:%d/%m/%Y} — "
      f"{len(grupos)} grupos ativos: {sorted(grupos)}")

## 4. A linha do tempo em texto

Uma linha por evento, ordenada. O símbolo mostra a precisão: `●` segundo · `◆` dia · `◇` mês · `○` ano autodeclarado · `?` visto pela 1ª vez.

In [ ]:
SIMBOLO = {"segundo": "●", "dia": "◆", "mês": "◇", "ano (autodeclarado)": "○", "visto pela 1ª vez": "?"}
for data, precisao, titulo, texto, grupo in sorted(EVENTOS):
    print(f"{data[:10]:<11}{SIMBOLO[precisao]} [{grupo:<15}] {titulo}")

print(f"\n✍️ Duas frases (Passo 7) sobre a janela {ini:%d/%m} a {fim_:%d/%m}: o que se agrupa ali")
print("   e o evento externo que o explica (aqui, o próprio golpe: disparos de 05/out, registro #3).")
print("\n✍️ Passo 8 ampliado nas notas: não segui, curti, comentei, denunciei nem enviei mensagem a nenhuma conta.")

In [ ]:
try:
    from google.colab import files
    files.download("timeline-isca.csv")
except ImportError:
    print("(Fora do Colab: timeline-isca.csv ficou na pasta. Importe na cópia do modelo do TimelineJS.)")

### 🧠 Camada 8

"Achadinhos BH" juntou seguidores reais; em 4/out, essa **prova social** (Cialdini) foi herdada pela Serra Azul falsa. E cuidado com a **correlação ilusória**: datas próximas só viram assinatura com um evento externo, com fonte, que as explique.

## ✅ Antes de fechar o Dia 25

- [ ] Consultas rodadas à mão, só lendo
- [ ] `timeline-isca.csv` importado no TimelineJS e publicado (12+ eventos, 3+ grupos)
- [ ] O WhatsApp como "visto pela 1ª vez"
- [ ] As duas frases do agrupamento, com fonte
- [ ] Passo 8 ampliado nas notas

**Amanhã (Dia 26):** o que foi apagado. A página "Segurança" da cooperativa mudou depois do golpe, e o site falso também. A Wayback lembra.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>